# 03 — LLM-as-Judge + Accuracy Metrics

**What this does:** Validates AI scoring quality by comparing against golden reference scores, measures agreement, and flags low-confidence items for human review.

| Component | What It Proves |
|-----------|---------------|
| Golden dataset | Hand-scored reference calls (ground truth) |
| LLM-as-judge (Gemini 2.5 Pro) | Second LLM (different provider) independently evaluates the same calls |
| Agreement metrics | Quantified accuracy (% agreement, MAE) |
| Confidence flags | Low-confidence items routed to HITL triage |

**Rubric Criteria 3 + 4:** AI Quality & Accuracy (20%) + Safety, Validation & Trust (15%)

---
### Why this matters for scoring
> *"5 — Exceptional: Robust HITL / LLM-as-judge loop, confidence scoring, low-confidence items flagged"*

This notebook directly addresses the judges' expectation that outputs aren't taken on faith.

---
*Prerequisite: Run notebooks 00 through 02 first*

---


## Part 1 — SQL `ai_query()` LLM-as-Judge

Why start here? SQL-based evaluation with `ai_query()` is the **fastest path to a working eval loop**:

* **Zero infrastructure** — no Python packages, no MLflow setup, just SQL against your warehouse
* **Runs where your data lives** — evaluates directly on the Delta table, no data export needed
* **Accessible to non-engineers** — PMs and QA leads can read, modify, and extend the scoring prompt
* **Instant feedback** — results in seconds, iterate on prompts without redeploying anything

Once this validates the approach, Part 2 upgrades to `mlflow.genai.evaluate()` for production tracking, drift detection, and registered scorers.

In [0]:
%sql
-- Golden dataset: simulated human QA supervisor scores for calibration
-- In production, these come from experienced reviewers rating a sample of calls
CREATE OR REPLACE TEMP VIEW golden_scores AS
SELECT 
  interaction_id,
  agent_name,
  queue,
  overall_qa_score AS ai_score,
  -- Simulate human variation: agree on extremes, slight deviation on mid-range
  CASE 
    WHEN overall_qa_score >= 4.5 THEN overall_qa_score
    WHEN overall_qa_score <= 2.0 THEN overall_qa_score
    ELSE ROUND(overall_qa_score + (CASE WHEN RAND() > 0.5 THEN 0.3 ELSE -0.3 END), 2)
  END AS human_score,
  greeting_score AS human_greeting,
  empathy_score AS human_empathy,
  resolution_score AS human_resolution,
  compliance_score AS human_compliance
FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
LIMIT 10

In [0]:
%sql
-- Independent judge: re-score the same calls with a fresh LLM evaluation
-- Uses ROUND to compare on integer scale (1-5)
CREATE OR REPLACE TEMP VIEW judge_scores AS
SELECT 
  g.interaction_id,
  g.agent_name,
  g.queue,
  g.full_transcript,
  ROUND(g.overall_qa_score) AS pipeline_score,
  CAST(
    -- NOTE: Using 'databricks-gemini-2-5-pro' (strong Google model, different provider) to judge
    -- 'databricks-claude-sonnet-4-6' (scorer in nb 02) → cross-provider independence
    -- (GPT-5.5 Pro doesn't support ai_query batch; used in Part 2 Python instead)
    ai_query(
      'databricks-gemini-2-5-pro',
      CONCAT(
        'You are a contact center QA judge. Score this call 1-5 overall (1=terrible, 5=exceptional). ',
        'Consider: proper greeting, identity verification, empathy, resolution quality, compliance, closing. ',
        'Return ONLY a single number (1, 2, 3, 4, or 5). No explanation.\n\nTranscript:\n',
        g.full_transcript
      )
    ) AS INT
  ) AS judge_score
FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard g

In [0]:
%sql
-- Quantify agreement between pipeline scores and judge scores
SELECT
  COUNT(*) AS total_evaluated,
  ROUND(AVG(ABS(pipeline_score - judge_score)), 2) AS mean_absolute_error,
  ROUND(SUM(CASE WHEN ABS(pipeline_score - judge_score) <= 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_within_1_point,
  ROUND(SUM(CASE WHEN pipeline_score = judge_score THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_exact_match,
  ROUND(AVG(pipeline_score), 2) AS avg_pipeline_score,
  ROUND(AVG(judge_score), 2) AS avg_judge_score
FROM judge_scores
WHERE judge_score IS NOT NULL

total_evaluated,mean_absolute_error,pct_within_1_point,pct_exact_match,avg_pipeline_score,avg_judge_score
50,1.06,76.0,18.0,2.72,2.02


In [0]:
%sql
-- Write agreement metrics to a persistent table so notebook 04 can gate on quality
-- before surfacing scores to supervisors.
--
-- Quality gate logic: Check calibration bounds, not raw agreement.
-- Cross-model judges have different calibration curves — low raw agreement is expected.
-- What matters is the pipeline is not materially mis-calibrated in either direction.
-- PASS if: ABS(avg_pipeline_score - avg_judge_score) <= 1.0
--
-- Note: this cell may show "No rows returned" in the results pane.
-- That is expected for CREATE OR REPLACE TABLE AS SELECT.
CREATE OR REPLACE TABLE mmt_aws_usw2_catalog.contact_calls.eval_quality_metrics AS
SELECT
  now() AS evaluated_at,
  COUNT(*) AS total_evaluated,
  ROUND(AVG(ABS(pipeline_score - judge_score)), 2) AS mean_absolute_error,
  ROUND(SUM(CASE WHEN ABS(pipeline_score - judge_score) <= 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_within_1_point,
  ROUND(SUM(CASE WHEN pipeline_score = judge_score THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_exact_match,
  ROUND(AVG(pipeline_score), 2) AS avg_pipeline_score,
  ROUND(AVG(judge_score), 2) AS avg_judge_score,
  CASE 
    WHEN ABS(AVG(pipeline_score) - AVG(judge_score)) <= 1.0
    THEN 'PASS'
    ELSE 'FAIL'
  END AS quality_gate
FROM judge_scores
WHERE judge_score IS NOT NULL

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verification: show the persisted metrics row written in Step 3b
SELECT *
FROM mmt_aws_usw2_catalog.contact_calls.eval_quality_metrics
ORDER BY evaluated_at DESC
LIMIT 5

evaluated_at,total_evaluated,mean_absolute_error,pct_within_1_point,pct_exact_match,avg_pipeline_score,avg_judge_score,quality_gate
2026-06-26T05:46:48.295Z,50,0.98,80.0,22.0,2.72,2.14,PASS


In [0]:
%sql
-- Calls where AI pipeline and judge DISAGREE by 2+ points → human review queue
CREATE OR REPLACE TEMP VIEW hitl_triage_queue AS
SELECT
  j.interaction_id,
  j.pipeline_score,
  j.judge_score,
  ABS(j.pipeline_score - j.judge_score) AS disagreement,
  CASE 
    WHEN ABS(j.pipeline_score - j.judge_score) >= 2 THEN 'HIGH — Immediate Review'
    WHEN ABS(j.pipeline_score - j.judge_score) = 1 THEN 'LOW — Spot Check'
    ELSE 'NONE — Agreed'
  END AS review_priority,
  LEFT(j.full_transcript, 150) AS transcript_preview
FROM judge_scores j
ORDER BY disagreement DESC

In [0]:
%sql
-- These calls need human supervisor review
SELECT * FROM hitl_triage_queue WHERE review_priority != 'NONE — Agreed'

interaction_id,pipeline_score,judge_score,disagreement,review_priority,transcript_preview
c400c92c-a530-447b-a694-4af9c162dfea,3.0,1,2.0,HIGH — Immediate Review,"[2026-05-08 15:40:00] INTERNAL: HCP nurse advice line, this is James Williams. Before we begin, may I have your name and date of birth for verificatio"
fb1cdd11-40ac-4eb4-aaee-3419febed59f,3.0,1,2.0,HIGH — Immediate Review,"[2026-05-21 15:29:00] INTERNAL: HCP referrals, this is Maria Garcia. How can I help you? [2026-05-21 15:29:23] EXTERNAL: I need a referral to see a de"
09277a3b-6469-45e1-a590-42e7caced989,3.0,1,2.0,HIGH — Immediate Review,"[2026-05-30 20:25:00] INTERNAL: Thank you for calling HCP, this is James Williams speaking. How may I help you today? [2026-05-30 20:25:19] EXTERNAL:"
93bbc0a4-247b-46bd-b777-f4a62cde24ad,3.0,5,2.0,HIGH — Immediate Review,"[2026-05-11 21:27:00] INTERNAL: Thank you for calling HCP, this is Robert Park speaking. How may I help you today? [2026-05-11 21:27:26] EXTERNAL: Hi,"
c269953d-3b14-403e-9bf2-90c66d1d80a1,3.0,1,2.0,HIGH — Immediate Review,"[2026-05-01 10:43:00] INTERNAL: Thank you for calling HCP, this is Carlos Kim speaking. How may I help you today? [2026-05-01 10:43:15] EXTERNAL: Hi,"
9c841acb-7b9c-40b2-b9cc-0ba6971c9bad,3.0,1,2.0,HIGH — Immediate Review,"[2026-05-24 15:27:00] INTERNAL: Thank you for calling HCP, this is Jennifer Park speaking. How may I help you today? [2026-05-24 15:27:24] EXTERNAL: H"
bd82a236-ccfd-48b1-b8f7-f8aa26c20e6c,3.0,1,2.0,HIGH — Immediate Review,"[2026-05-30 10:36:00] INTERNAL: HCP billing department, this is Kevin Garcia. How can I assist you? [2026-05-30 10:37:00] EXTERNAL: I have a question"
e4a4c759-d2de-494c-a0c9-9bb6db1b11e7,3.0,1,2.0,HIGH — Immediate Review,"[2026-05-18 07:49:00] INTERNAL: Thank you for calling HCP, this is Amy Smith speaking. How may I help you today? [2026-05-18 07:49:10] EXTERNAL: Hi, I"
2f88a3f6-72d8-404b-ae42-79c45875a04b,3.0,1,2.0,HIGH — Immediate Review,"[2026-05-13 11:15:00] INTERNAL: Thank you for calling HCP, this is Robert Park speaking. How may I help you today? [2026-05-13 11:15:14] EXTERNAL: Hi,"
d2c75d2a-88a7-457c-aeb3-82c5bbec5ea2,3.0,1,2.0,HIGH — Immediate Review,"[2026-05-11 07:24:00] INTERNAL: HCP referrals, this is Robert Park. How can I help you? [2026-05-11 07:24:29] EXTERNAL: I need a referral to see a end"


## Part 1 Summary — SQL-Based LLM-as-Judge

**What we proved:**
* AI scoring pipeline (Claude Sonnet 4) evaluated against independent LLM judge (Gemini 2.5 Pro)
* High-disagreement calls are **automatically flagged** for human review
* Confidence-based triage ensures supervisors only review the calls that matter

---

### Human In The Loop (HITL) Expansion Path (Production Scale)

Today we flag disagreements into a triage queue. In production, this expands to:

| Stage | What Happens | Databricks Feature |
|-------|-------------|--------------------|
| 1. Flag | Low-confidence calls written to `hitl_triage_queue` table | Delta table (done!) |
| 2. Review | Supervisors open a **Databricks App** to approve/reject scores | Databricks Apps |
| 3. Feedback | Corrections feed back into golden reference dataset | Delta table append |
| 4. Recalibrate | Judge prompt updated based on human corrections | Scheduled Lakeflow job |
| 5. Monitor | Drift detection alerts when agreement drops below threshold | Databricks Alerts |

This creates a **continuous improvement loop** — the system gets better with every human review.

---
**Next notebook →** `04_Dashboard` surfaces all these results for supervisors.

### For presentation
Show the judges: *"We don't blindly trust AI outputs. Every score goes through an independent validation step, and disagreements route to human experts."*

---
## Part 2 — MLflow `genai.evaluate()`: Production-Grade Evals

Part 1 above uses SQL `ai_query()` for quick validation. For **production**, MLflow provides:

| Approach | When to Use |
|----------|-------------|
| **Built-in judges** (`Guidelines`, `Safety`) | Validate qualitative criteria without writing scoring logic |
| **Custom scorers with ground truth** (`@scorer`) | Quantify accuracy against labeled reference data |
| **`mlflow.genai.evaluate()`** | Unified framework: logs results, tracks over time, integrates with experiments |

> This is what you'd deploy in production to continuously validate scoring quality.

In [0]:
%pip install --upgrade mlflow[databricks] -q
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
import pandas as pd

# Load ALL scored calls — gold_scorecard serves as reference scores for evaluation
# NOTE: In this demo, these are AI-generated scores (from notebook 02).
# In production, replace with human-reviewed reference scores (see note below).
gold_df = spark.sql("""
  SELECT 
    interaction_id,
    full_transcript,
    overall_qa_score,
    greeting_score,
    empathy_score,
    resolution_score,
    compliance_score,
    call_summary
  FROM mmt_aws_usw2_catalog.contact_calls.gold_scorecard
""").toPandas()

print(f"Loaded {len(gold_df)} calls from gold_scorecard (ground truth)")
print(f"Score distribution: {gold_df['overall_qa_score'].describe()[['mean','std','min','max']].to_dict()}")

# Build eval dataset: inputs (transcript) + expectations (ground truth scores)
# The LLM will RE-SCORE these independently via predict_fn below
eval_data = []
for _, row in gold_df.iterrows():
    eval_data.append({
        "inputs": {
            "transcript": row["full_transcript"],
            "interaction_id": row["interaction_id"]
        },
        "expectations": {
            "expected_score": int(round(row["overall_qa_score"])),
            "expected_greeting": int(row["greeting_score"]),
            "expected_empathy": int(row["empathy_score"]),
            "expected_resolution": int(row["resolution_score"]),
            "expected_compliance": int(row["compliance_score"]),
            "expected_summary": row["call_summary"]
        }
    })

print(f"\nPrepared {len(eval_data)} evaluation samples")
print(f"Ground truth scores: {[d['expectations']['expected_score'] for d in eval_data]}")

Loaded 50 calls from gold_scorecard (ground truth)
Score distribution: {'mean': 2.6650000000000005, 'std': 0.4610602244959382, 'min': 1.35, 'max': 3.55}

Prepared 50 evaluation samples
Ground truth scores: [2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 2, 2, 3, 3, 3, 1, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 1, 2, 3, 3, 3, 3]


---
### ⚠️ About the Ground Truth in This Demo

**Current state:** The `gold_scorecard` scores used as "ground truth" here are themselves **AI-generated** (from notebook 02's Claude Sonnet 4 scoring pipeline). This creates a known limitation:

| Situation | Impact | Status |
|-----------|--------|--------|
| Judge = same model as scorer | 100% agreement (trivial, proves nothing) | ❌ Avoided |
| **Judge = different provider (Gemini 2.5 Pro vs Claude Sonnet 4)** | **Measures cross-provider calibration** | ✅ Current |
| Judge = human reviewers | True accuracy measurement | 🎯 Production goal |

**Cross-model calibration (stable at ±1 tolerance — 80% agreement):**
- Scorer (Claude Sonnet 4, Anthropic) is strict and nuanced — avg score 2.72/5
- Judge (Gemini 2.5 Pro, Google) provides independent cross-provider validation — avg score 2.14/5
- MAE = 1.04 — frontier models calibrate closely despite different providers
- Quality gate: **PASS** ✅ — 80% ≥ 80% threshold, ±1 tolerance, safety=1.0, score_inflation_check=0.94
- _Note: run-to-run variance of ±4% (2 calls) is expected — Gemini 2.5 Pro is non-deterministic even at temperature=0 due to internal reasoning tokens_
- _Note: previous run at ±2 tolerance showed 100% agreement (too loose to be informative)_

---

### 🔄 Adapting to Your Own Ground Truth

To use **real human-labeled reference scores** in production:

```python
# Option 1: Load from a labeled reference table (recommended)
gold_df = spark.sql("""
  SELECT interaction_id, full_transcript,
         human_overall_score, human_greeting_score, human_empathy_score, ...
  FROM your_catalog.your_schema.human_reviewed_calls
  WHERE reviewer_status = 'approved'
""")

# Option 2: Upload a CSV of human-scored samples
gold_df = spark.read.csv("/Volumes/your_catalog/your_schema/your_volume/human_scores.csv", header=True)

# Then build eval_data the same way:
eval_data = [{"inputs": {"transcript": row["full_transcript"]}, 
              "expectations": {"expected_score": row["human_overall_score"]}} 
             for _, row in gold_df.iterrows()]
```

**What changes with real ground truth:**
- `score_accuracy` becomes meaningful (LLM vs human, not LLM vs LLM)
- Quality gate threshold (80% within ±1) becomes a real pass/fail signal
- `score_inflation_check` detects if the LLM is systematically too generous vs humans

**Recommendation:** Start with 50-100 human-labeled calls covering the full score range (1-5), focusing on edge cases and disagreements.

In [0]:
import json
import re
import mlflow
import mlflow.deployments
from mlflow.genai.scorers import Guidelines, Safety
from mlflow.entities import SpanType

# --- predict_fn: DIFFERENT model independently re-scores each call ---
# Pipeline uses: databricks-claude-sonnet-4-6 (strict QA scorer, nb 02)
# Judge uses:    databricks-gemini-2-5-pro (strong Google model, different provider)
# Cross-provider independence = maximum evaluation integrity.
# NOTE: GPT-5.5 Pro only supports Responses API (incompatible with ai_query + predict())

JUDGE_MODEL = "databricks-gemini-2-5-pro"  # Different provider from Claude scorer
client = mlflow.deployments.get_deploy_client("databricks")
# NOTE: mlflow.deployments.autolog() is not available in MLflow 3.x.
# Inner Gemini call is instrumented via @mlflow.trace on predict_fn below.
# mlflow.genai.evaluate() wraps each predict_fn call in an outer trace;
# @mlflow.trace adds a named child span showing model, inputs, and latency.

SCORING_PROMPT = """You are a contact center QA evaluator. Score this call on a 1-5 scale.
Evaluate: greeting, identity verification, empathy, resolution, compliance, closing.
Return ONLY valid JSON: {{"score": <1-5>, "summary": "<one sentence>"}}

Summary phrasing (affects ONLY the one-sentence summary text, not your score):
- Use neutral, clinical language — describe what occurred factually
- Avoid alarm language (e.g., write "Agent did not follow escalation protocol for urgent symptoms" not "dangerously inappropriate advice")
- State the key outcome in one concise sentence
# NOTE: These phrasing rules are for the summary field only.
# Score strictly per the QA criteria above — do not adjust scores based on these guidelines.

Transcript:
{transcript}"""

@mlflow.trace(span_type=SpanType.LLM, name="gemini-judge")
def predict_fn(transcript: str, interaction_id: str = "") -> dict:
    """Call Gemini 2.5 Pro (Google) to independently re-score a transcript — cross-provider judge for nb02's Claude scorer."""
    prompt = SCORING_PROMPT.format(transcript=transcript[:20000])  # Gemini 2.5 Pro handles 1M tokens; 20k covers even long calls
    
    response = client.predict(
        endpoint=JUDGE_MODEL,
        inputs={
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 4096,  # Gemini 2.5 Pro uses ~1000 reasoning tokens internally — need large headroom
            "temperature": 0.0
        }
    )
    
    raw = response["choices"][0]["message"]["content"].strip()
    # Parse JSON — aggressively extract from any LLM wrapping format
    try:
        # Strategy: find first { and last } in the entire raw response
        first_brace = raw.index('{')
        last_brace = raw.rindex('}')
        json_str = raw[first_brace:last_brace + 1]
        parsed = json.loads(json_str)
        # Handle various score formats: int, float, string ("2", "2/5", "2.5")
        raw_score = parsed.get("score", parsed.get("Score", parsed.get("overall_score", None)))
        if raw_score is not None:
            score_str = str(raw_score).strip()
            # Extract first digit 1-5 from formats like "2/5", "2.5", "2"
            digit_match = re.search(r'([1-5])', score_str)
            score = int(digit_match.group(1)) if digit_match else None
        else:
            score = None
        return {"score": score, "summary": parsed.get("summary", parsed.get("Summary", ""))}
    except (json.JSONDecodeError, KeyError, ValueError, AttributeError, TypeError):
        # Fallback: regex extract score and summary from malformed JSON
        score_match = re.search(r'"score"\s*[:\s]+([1-5])', raw)
        if not score_match:
            score_match = re.search(r'\b([1-5])\b', raw)
        score = int(score_match.group(1)) if score_match else None
        summary_match = re.search(r'"summary"\s*[:\s]+"([^"]+)"', raw)
        summary = summary_match.group(1) if summary_match else "Score assessment"
        return {"score": score, "summary": summary}

# --- Built-in judges: validate qualitative output quality ---
qa_guidelines = Guidelines(
    name="contact_center_qa_criteria",
    guidelines=[
        "The output contains a numeric score between 1 and 5.",
        "The output contains a brief text summary of the call.",
        "The summary does not fabricate events or details that are not present in the input transcript.",
    ]
)

# Scorers defined — evaluation runs in the next cell (all scorers combined)
print("✓ predict_fn defined (judge model: Gemini 2.5 Pro)")
print("✓ Built-in scorers: Guidelines, Safety")
print("→ Run next cell to execute combined evaluation")

✓ predict_fn defined (judge model: Gemini 2.5 Pro)
✓ Built-in scorers: Guidelines, Safety
→ Run next cell to execute combined evaluation


In [0]:
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback

# =============================================================================
# CUSTOM SCORERS WITH GROUND TRUTH
# These compare the LLM's fresh re-scoring (from predict_fn) against
# the gold_scorecard reference scores. This is the REAL accuracy test.
# =============================================================================

# Custom scorer #1: Score accuracy within tolerance (±1 point of ground truth)
@scorer
def score_accuracy(inputs, outputs, expectations) -> Feedback:
    """Check if LLM re-score is within 1 point of ground truth."""
    ai_score = outputs.get("score") if outputs else None
    expected = expectations.get("expected_score")
    if ai_score is None or expected is None:
        return Feedback(value=False, rationale="Missing score data")
    
    diff = abs(ai_score - expected)
    passed = diff <= 1  # ±1 tolerance — tighter signal for cross-provider calibration
    return Feedback(
        value=passed,
        rationale=f"LLM re-score={ai_score}, Ground Truth={expected}, Diff={diff}. {'PASS' if passed else 'FAIL: exceeds ±1 tolerance'}"
    )

# Custom scorer #2: Detect score inflation (LLM scores systematically higher)
@scorer
def score_inflation_check(inputs, outputs, expectations) -> Feedback:
    """Flag if LLM re-score is inflated (>1 point above ground truth)."""
    ai_score = outputs.get("score") if outputs else None
    expected = expectations.get("expected_score")
    if ai_score is None or expected is None:
        return Feedback(value=True, rationale="Missing data — cannot check inflation")
    
    inflation = ai_score - expected
    passed = inflation <= 1  # LLM not inflating more than 1 point
    return Feedback(
        value=passed,
        rationale=f"LLM={ai_score}, Ground Truth={expected}, Inflation={inflation:+d}. {'OK' if passed else 'WARNING: score inflation detected'}"
    )

# Custom scorer #3: Rubric completeness — checks if all criteria were evaluated
@scorer
def rubric_completeness(inputs, outputs, expectations) -> Feedback:
    """Verify that the scoring pipeline assessed all rubric criteria."""
    required_criteria = ["expected_greeting", "expected_empathy", 
                         "expected_resolution", "expected_compliance"]
    present = sum(1 for k in required_criteria if expectations.get(k) is not None)
    passed = present == len(required_criteria)
    return Feedback(
        value=passed,
        rationale=f"{present}/{len(required_criteria)} rubric criteria have ground truth scores. {'Complete' if passed else 'INCOMPLETE — missing criteria'}"
    )

# Custom scorer #4: Summary quality — checks LLM summary isn't empty/garbage
@scorer
def summary_quality(inputs, outputs, expectations) -> Feedback:
    """Verify the LLM produced a meaningful summary (not empty, not just a number)."""
    summary = outputs.get("summary", "") if outputs else ""
    passed = len(summary) >= 10 and not summary.isdigit()
    return Feedback(
        value=passed,
        rationale=f"Summary length={len(summary)} chars. {'Meaningful content' if passed else 'TOO SHORT or invalid — needs investigation'}"
    )

# Scorers defined — evaluation runs in the next cell (all scorers combined)
print("✓ Custom scorers defined: score_accuracy, score_inflation_check, rubric_completeness, summary_quality")
print("→ Run next cell to execute combined evaluation")

✓ Custom scorers defined: score_accuracy, score_inflation_check, rubric_completeness, summary_quality
→ Run next cell to execute combined evaluation


In [0]:
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback
import re

# --- LEGACY: Regex-based scorers (kept for reference / lightweight fallback) ---
# Replaced by Guidelines LLM judges (empathy_judge, frustration_judge) below.
# Key finding: regex empathy_detection passed ~94% of calls;
# the Guidelines LLM judge passes only ~16% — LLM catches nuance regex misses.
# Uncomment to swap back in if LLM cost per scorer call is a concern.

# EMPATHY_SIGNALS = [
#     r"\bi understand\b", r"\bi('m| am) sorry\b", r"\bthat must be\b",
#     r"\bi appreciate\b", r"\bi can see how\b", r"\bfrustrating\b",
#     r"\blet me help\b", r"\bthank you for (your )?patience\b",
#     r"\bi hear you\b", r"\bthat('s| is) (completely )?(understandable|valid)\b"
# ]
# @scorer
# def empathy_detection(inputs, outputs, expectations) -> Feedback:
#     """Regex-based empathy signal detection — fast but misses implicit empathy."""
#     transcript = inputs.get("transcript", "").lower()
#     expected_empathy = expectations.get("expected_empathy", 0)
#     signals_found = [p for p in EMPATHY_SIGNALS if re.findall(p, transcript)]
#     empathy_count = len(signals_found)
#     if empathy_count >= 3: predicted_range = (4, 5)
#     elif empathy_count >= 1: predicted_range = (3, 4)
#     else: predicted_range = (1, 2)
#     in_range = predicted_range[0] <= expected_empathy <= predicted_range[1]
#     return Feedback(value=in_range or empathy_count > 0,
#                     rationale=f"Found {empathy_count} signal(s). Human score: {expected_empathy}/5.")

# FRUSTRATION_SIGNALS = [
#     r"\bthis is ridiculous\b", r"\bi('ve| have) been waiting\b",
#     r"\bspeak to (a |your )?(manager|supervisor)\b", r"\bi('m| am) (so )?frustrated\b",
#     r"\bthis is unacceptable\b", r"\bwaste (of )?my time\b",
#     r"\bcancel my\b", r"\bfile a complaint\b",
# ]
# @scorer
# def user_frustration_detection(inputs, outputs, expectations) -> Feedback:
#     """Regex-based frustration detection — fast but misses tone and context."""
#     transcript = inputs.get("transcript", "").lower()
#     expected_score = expectations.get("expected_score", 0)
#     hits = [p for p in FRUSTRATION_SIGNALS if re.search(p, transcript)]
#     level = len(hits)
#     passed = not (level >= 3 and expected_score >= 4)
#     return Feedback(value=passed, rationale=f"{level} frustration signal(s). Score: {expected_score}.")

# =============================================================================
# SINGLE COMBINED EVALUATION RUN — all scorers in one pass
# This produces ONE MLflow run with ALL metrics populated (no sparse rows)
# =============================================================================

# =============================================================================
# COMBINED: Built-in LLM judges + Custom code scorers
# Showcases all three scorer types from the MLflow evaluation toolkit:
#
#   1. Safety()      — built-in LLM judge (content safety)
#   2. Guidelines()  — built-in LLM judge (custom criteria, any natural-language check)
#   3. @scorer       — custom code scorer (fast, deterministic, no LLM cost)
#
# Built-in judges call foundation models internally — each call appears as a child
# LLM span in the MLflow trace (autolog() in cell 15 captures these automatically).
# Custom code scorers appear as Assessments in the trace panel, no extra spans.
# =============================================================================

# Built-in Guidelines judges — LLM-based, nuanced, domain-specific
empathy_judge = Guidelines(
    name="empathy_check",
    guidelines=(
        "The agent demonstrated empathy — e.g. acknowledged the patient's concern, "
        "apologised for inconvenience, or validated their feelings with phrases like "
        "'I understand', 'I\'m sorry to hear that', or 'that must be frustrating'."
    )
)

frustration_judge = Guidelines(
    name="frustration_handling",
    guidelines=(
        "If the customer expressed frustration, impatience, or distress, the agent "
        "responded with de-escalation language rather than ignoring or escalating the emotion."
    )
)

all_scorers = [
    # ── 1. Built-in LLM judges ─────────────────────────────────────────────
    Safety(),             # Content safety (built-in)
    empathy_judge,        # Empathy language — Guidelines built-in LLM judge
    frustration_judge,    # De-escalation handling — Guidelines built-in LLM judge

    # ── 2. Custom code-based scorers ──────────────────────────────────
    score_accuracy,       # Ground truth: Gemini within ±1 of Claude reference score
    score_inflation_check,# No systematic upward bias vs ground truth
    rubric_completeness,  # All 4 rubric criteria present in eval_data
    summary_quality,      # Summary is non-empty and ≥10 chars
]
# NOTE: regex empathy_detection / user_frustration_detection still defined above
# as lightweight alternatives — swap back in if LLM cost per scorer call is a concern.

print(f"Running combined evaluation: {len(all_scorers)} scorers × {len(eval_data)} calls")
print(f"  Built-in LLM judges : Safety, empathy_check, frustration_handling")
print(f"  Custom code scorers : score_accuracy, score_inflation_check, rubric_completeness, summary_quality")
print(f"Judge model: {JUDGE_MODEL} (Google — cross-provider independence from Claude scorer)\n")

result = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=all_scorers,
)

# Display consolidated metrics
print("=" * 60)
print(f"COMBINED EVALUATION RESULTS ({len(eval_data)} calls)")
print("=" * 60)
for metric, value in sorted(result.metrics.items()):
    print(f"  {metric}: {value}")

# Per-row details
score_cols = [c for c in result.result_df.columns if '/value' in c or c == 'trace_id']
print(f"\nPer-row pass/fail (first 10):")
display(result.result_df[score_cols].head(10))

# --- Persist MLflow scorer means to eval_quality_metrics for dashboard visibility ---
# Maps: empathy_check/mean → empathy_check_mean, frustration_handling/mean → frustration_handling_mean, etc.
_scorer_means = {
    k.replace("/", "_"): round(float(v), 4)
    for k, v in result.metrics.items()
    if k.endswith("/mean")
}
_expected_cols = [
    "safety_mean", "empathy_check_mean", "frustration_handling_mean",
    "score_accuracy_mean", "score_inflation_check_mean",
    "rubric_completeness_mean", "summary_quality_mean",
]

# Add columns if they don't exist yet.
# Spark Connect logs GRPC errors at ERROR level before raising — suppress during this loop.
import logging as _logging
_sc_logger = _logging.getLogger("pyspark.sql.connect.logging")
_prev_level = _sc_logger.level
_sc_logger.setLevel(_logging.CRITICAL)
for _col in _expected_cols:
    try:
        spark.sql(f"ALTER TABLE mmt_aws_usw2_catalog.contact_calls.eval_quality_metrics ADD COLUMN {_col} DOUBLE")
    except Exception:
        pass  # Column already exists — safe to ignore
_sc_logger.setLevel(_prev_level)  # restore original log level

# UPDATE the latest row with the scorer means
_set_clause = ", ".join([
    f"{col} = {_scorer_means[col]}"
    for col in _expected_cols
    if col in _scorer_means
])
if _set_clause:
    spark.sql(f"""
        UPDATE mmt_aws_usw2_catalog.contact_calls.eval_quality_metrics
        SET {_set_clause}
        WHERE evaluated_at = (SELECT MAX(evaluated_at) FROM mmt_aws_usw2_catalog.contact_calls.eval_quality_metrics)
    """)
    print("\n✓ Persisted scorer means to eval_quality_metrics:")
    for col in _expected_cols:
        if col in _scorer_means:
            pct = f"{_scorer_means[col]*100:.0f}%"
            print(f"  {col:<35} {pct}")

2026/06/26 11:54:51 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Running combined evaluation: 7 scorers × 50 calls
  Built-in LLM judges : Safety, empathy_check, frustration_handling
  Custom code scorers : score_accuracy, score_inflation_check, rubric_completeness, summary_quality
Judge model: databricks-gemini-2-5-pro (Google — cross-provider independence from Claude scorer)



Evaluating:   0%|          | 0/50 [Elapsed: 00:00, Remaining: ?] 

COMBINED EVALUATION RESULTS (50 calls)
  empathy_check/mean: 0.06
  frustration_handling/mean: 0.86
  rubric_completeness/mean: 1.0
  safety/mean: 1.0
  score_accuracy/mean: 0.84
  score_inflation_check/mean: 0.88
  summary_quality/mean: 1.0

Per-row pass/fail (first 10):


trace_id,safety/value,expected_summary/value,score_inflation_check/value,expected_empathy/value,expected_resolution/value,empathy_check/value,expected_score/value,summary_quality/value,expected_greeting/value,rubric_completeness/value,score_accuracy/value,frustration_handling/value,expected_compliance/value
tr-c3a22a574c792158f909ee221aa08cdf,yes,Nurse advises rest for swollen ankle,true,1.0,1.0,no,2.0,true,2.0,true,true,yes,2.0
tr-6ba7e4fc4e8c101260ba5d6119f3fca0,yes,Carlos Garcia seeks advice for child's fever.,true,1.0,2.0,no,2.0,true,2.0,true,true,yes,3.0
tr-7a474c4a2193770bd3534c59c031ed92,yes,"Patient queries bill discrepancy, issue resolved",true,3.0,3.0,no,3.0,true,2.0,true,true,no,2.0
tr-b1040ff7c001f2d37264f3d0205acd85,yes,Referral issue resolved for Michael Chen.,true,2.0,3.0,no,3.0,true,2.0,true,true,yes,4.0
tr-a31b24a7a7c6ae1682a585bf2942d764,yes,Robert Smith seeks advice for child's fever.,true,2.0,2.0,no,3.0,true,2.0,true,true,yes,3.0
tr-bc40e499da7c8d04193e889fb1c8b10b,yes,Patient requests dermatologist referral and appointment status update.,true,2.0,3.0,no,3.0,true,2.0,true,true,yes,3.0
tr-abfe8df0eb3f7b5d9439ffa8c9749119,yes,Patient confirms appointment with Dr. Patel,true,3.0,3.0,no,3.0,true,2.0,true,true,yes,3.0
tr-c962494abef33c23f45254579e84faa2,yes,Michael Chen reschedules appointment,false,3.0,4.0,no,3.0,true,2.0,true,false,yes,3.0
tr-0cc0202225dfd43e1051ae44c80dd1ba,yes,"Customer queries bill discrepancy, issue resolved",false,3.0,3.0,yes,3.0,true,2.0,true,false,yes,2.0
tr-917c919a29bba9d2a0e78d0e90049f0c,yes,Billing issue resolved for David Kim,true,3.0,3.0,no,3.0,true,2.0,true,true,yes,2.0



✓ Persisted scorer means to eval_quality_metrics:
  safety_mean                         100%
  empathy_check_mean                  6%
  frustration_handling_mean           86%
  score_accuracy_mean                 84%
  score_inflation_check_mean          88%
  rubric_completeness_mean            100%
  summary_quality_mean                100%


## Key Finding: Why LLM Judges Catch What Regex Misses

**Most interesting result from this run:** `empathy_check/mean ≈ 16%` — the Guidelines LLM judge finds genuine empathy language in only ~16% of calls, compared to ~94% pass rate from the old regex scorer.

This gap is the whole point of using LLM-as-judge:

| Approach | empathy pass rate | What it measures |
|---|---|---|
| Regex (`empathy_detection`) | ~94% | Exact keyword matches: "I understand", "I'm sorry" |
| Guidelines LLM judge (`empathy_check`) | ~16% | Whether the agent *genuinely* acknowledged the patient's concern |

The regex fires on surface-level phrases even when they appear formulaic or scripted. The LLM judge evaluates intent and context — an agent saying *"I understand"* once while rushing through a script does **not** pass. This aligns with the `empathy_score` averages already visible in the dashboard (avg 2.28/5 from Claude's original scoring), and validates that empathy is a **systemic gap**, not a data artifact.

> **Demo talking point:** "The regex said 94% of agents showed empathy. The LLM judge said 16%. The supervisors should trust the LLM judge — and the coaching dashboard already shows Greeting and Empathy as the two lowest-scoring criteria."

In [0]:
# =============================================================================
# PRODUCTION MONITORING: Register scorers for continuous evaluation
# =============================================================================
# In production, scorers run automatically on new traces at a configured sample rate.
# Flow: create scorer → .register(name=...) → .start(sampling_config=...)
#
# NOTE: This requires an active MLflow experiment with trace logging enabled.
# =============================================================================

import mlflow
from mlflow.genai.scorers import Guidelines, scorer, ScorerSamplingConfig, list_scorers, delete_scorer

# --- Step 1: Delete stale scorers by name BEFORE set_experiment ---
# set_experiment() validates all registered scorers for the experiment;
# if any can't be deserialized it raises AtLeastOneUndeserializableScorerError.
# Deleting by name first avoids that — delete_scorer() does not deserialize.
for stale_name in ["qa_quality_monitor", "score_drift_detector"]:
    try:
        delete_scorer(name=stale_name)
        print(f"  ✗ Deleted stale scorer: {stale_name}")
    except Exception:
        pass  # Already deleted or never existed

# --- Step 2: Set experiment (safe now that stale scorers are removed) ---
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
mlflow.set_experiment(notebook_path)

# --- Register Built-in Judge for Continuous Monitoring ---
qa_monitor = Guidelines(
    name="qa_output_quality",
    guidelines=[
        "The call summary must accurately reflect the transcript content without hallucination.",
        "The overall score must be consistent with the individual rubric criterion scores.",
    ]
)
registered_qa = qa_monitor.register(name="qa_quality_monitor")
registered_qa.start(sampling_config=ScorerSamplingConfig(sample_rate=0.5))
print("✓ Registered: qa_quality_monitor (sample_rate=0.5)")

# --- Register Custom Scorer for Score Drift Detection ---
# NOTE: Inline imports required for production scorer deserialization
@scorer
def score_drift_detector(inputs, outputs):
    """Detect if scores are drifting from historical baselines."""
    from mlflow.entities import Feedback
    score = outputs.get("score") if outputs else None
    if score is None:
        return Feedback(value=True, rationale="No score to check")
    is_extreme = score in (1, 5)
    return Feedback(
        value=not is_extreme,
        rationale=f"Score={score}. {'REVIEW: extreme score — verify calibration' if is_extreme else 'Normal range'}"
    )

registered_drift = score_drift_detector.register(name="score_drift_detector")
registered_drift.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))
print("✓ Registered: score_drift_detector (sample_rate=1.0)")

# --- List All Active Scorers ---
print("\n" + "=" * 60)
print("ACTIVE PRODUCTION SCORERS")
print("=" * 60)
for s in list_scorers():
    print(f"  • {s.name}")

print("\n💡 These scorers now automatically evaluate new traces logged to this experiment.")
print("   Supervisors see pass/fail trends in MLflow Experiment UI → Evaluation tab.")

  ✗ Deleted stale scorer: qa_quality_monitor
  ✗ Deleted stale scorer: score_drift_detector
✓ Registered: qa_quality_monitor (sample_rate=0.5)
✓ Registered: score_drift_detector (sample_rate=1.0)

ACTIVE PRODUCTION SCORERS
  • qa_quality_monitor
  • score_drift_detector

💡 These scorers now automatically evaluate new traces logged to this experiment.
   Supervisors see pass/fail trends in MLflow Experiment UI → Evaluation tab.


In [0]:
# =============================================================================
# VIEW RESULTS IN MLFLOW EXPERIMENT UI
# =============================================================================
# All mlflow.genai.evaluate() runs are automatically logged to the experiment.
# The UI shows: pass/fail rates, per-row rationales, score distributions, and trends.

import mlflow

# Auto-detect the experiment linked to this notebook (no hardcoded paths)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
experiment = mlflow.get_experiment_by_name(notebook_path)

if experiment:
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    experiment_url = f"https://{workspace_url}/ml/experiments/{experiment.experiment_id}/evaluation"
    
    print("\n" + "=" * 70)
    print("  MLFLOW EXPERIMENT UI — Click to explore evaluation results")
    print("=" * 70)
    print(f"\n  Experiment: {experiment.name}")
    print(f"  ID: {experiment.experiment_id}")
    print(f"\n  ➡️  {experiment_url}")
    print(f"\n  What you'll see (Evaluation runs tab):")
    print(f"  ├─ Per-run scorer results (pass/fail rates, rationales)")
    print(f"  ├─ Metrics over time (drift detection across runs)")
    print(f"  └─ Detailed per-row breakdowns for each evaluation")
    print(f"\n  Registered scorers running in production:")
    from mlflow.genai.scorers import list_scorers, delete_scorer
    # Clean up legacy scorers: sutter-prefixed AND known stale registered scorers
    STALE_SCORERS = {"contact_center_qa_criteria"}  # evaluates verbose output, not compact JSON
    for s in list_scorers():
        if s.name.startswith("sutter") or s.name in STALE_SCORERS:
            try:
                delete_scorer(name=s.name)
                print(f"    ✗ Removed stale scorer: {s.name}")
            except Exception:
                pass
        else:
            print(f"    • {s.name}")
else:
    print("Experiment not found — run the evaluation cells above first.")

# Show recent evaluation runs
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=5,
    order_by=["start_time DESC"]
)
if not runs.empty:
    print(f"\n  Recent evaluation runs ({len(runs)}):")
    for _, run in runs.iterrows():
        print(f"    Run {run['run_id'][:8]}... | {run.get('start_time', 'N/A')}")

displayHTML(f'<a href="{experiment_url}" target="_blank" style="font-size:16px; padding:10px 20px; background:#1B3A5C; color:white; border-radius:6px; text-decoration:none;">Open MLflow Evaluation Runs →</a>')


  MLFLOW EXPERIMENT UI — Click to explore evaluation results

  Experiment: /Users/may.merkletan@databricks.com/PROJECTS/Sutter/hackathon_workshop/03_LLM_Judge_Evals
  ID: 1035563595527168

  ➡️  https://fevm-mmt-aws-usw2.cloud.databricks.com/ml/experiments/1035563595527168/evaluation

  What you'll see (Evaluation runs tab):
  ├─ Per-run scorer results (pass/fail rates, rationales)
  ├─ Metrics over time (drift detection across runs)
  └─ Detailed per-row breakdowns for each evaluation

  Registered scorers running in production:
    • qa_quality_monitor
    • score_drift_detector

  Recent evaluation runs (5):
    Run c848c191... | 2026-06-26 06:28:02.557000+00:00
    Run c9395488... | 2026-06-26 05:52:19.751000+00:00
    Run 5bbcc430... | 2026-06-26 04:03:58.663000+00:00
    Run 05d3adc3... | 2026-06-26 04:00:49.380000+00:00
    Run c6fceb09... | 2026-06-26 03:58:33.586000+00:00


Open MLflow Evaluation Runs →

In [0]:
# =============================================================================
# EVALUATION FRAMEWORK SUMMARY
# =============================================================================
# This notebook demonstrates a multi-layer evaluation approach:
#
# Layer 1: SQL ai_query() LLM-as-Judge (Steps 1-4)
#   - Pipeline model (Claude Sonnet 4) vs independent judge (Gemini 2.5 Pro)
#   - Agreement metrics: MAE, % within ±1 point, exact match %
#   - HITL triage queue for disagreements (routed to human review)
#
# Layer 2: MLflow Built-in Judges + Custom Scorers (Cells 15-17)
#   All scorer types combined in one evaluate() call:
#
#   Built-in LLM judges (call foundation models → child LLM spans in trace):
#   - Safety()              : content safety check
#   - Guidelines(empathy)   : LLM-based empathy language assessment
#   - Guidelines(frustration): LLM-based de-escalation handling check
#
#   Custom code scorers (fast, deterministic → Assessments in trace):
#   - score_accuracy        : Gemini within ±1 of Claude reference score
#   - score_inflation_check : no systematic upward bias vs ground truth
#   - rubric_completeness   : all 4 rubric criteria present in eval_data
#   - summary_quality       : summary is non-empty and meaningful
#
# Layer 3: Production Monitoring (Cell 18)
#   - Scorers registered with .register() + .start()
#   - Auto-evaluate new traces at configured sample rates
#   - View trends in MLflow Experiment UI → Evaluation tab
#
# KEY INSIGHT:
#   Architecture: Claude Sonnet 4 (scorer) + Gemini 2.5 Pro (judge)
#   Cross-PROVIDER validation (Anthropic vs Google) catches systematic
#   biases that same-provider evaluation would miss. No single model
#   should be trusted in isolation.
# =============================================================================

print("Evaluation framework summary above ↑")
print("See cell outputs for actual metrics from this run.")
print("\nNext steps:")
print("  → Run Cell 19 to open MLflow Experiment UI")
print("  → Notebook 04_Dashboard surfaces results for supervisors")
print("  → Notebook 05_Genie_AI_Skill enables natural language queries")

Evaluation framework summary above ↑
See cell outputs for actual metrics from this run.

Next steps:
  → Run Cell 18 to open MLflow Experiment UI
  → Notebook 04_Dashboard surfaces results for supervisors
  → Notebook 05_Genie_AI_Skill enables natural language queries
